# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`.

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure.

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

# Solution

## 1. Scraping plan

The website has two useful page types:

- **Listing pages** contain 20 book cards with the title, price, star-rating class, availability, and a link to the product page.
- **Product pages** contain the UPC, full availability text, genre breadcrumb, and optional description.

The scraper follows each listing page through its **next** link. It checks rating and price from the card first, and visits the detail page only for books that pass the filters. This avoids unnecessary requests.

We use:

- `requests` to download HTML,
- `BeautifulSoup` to select elements from the HTML,
- `urljoin` to turn relative links into reliable absolute URLs,
- `pandas` to return the requested table.

> The target is a purpose-built scraping sandbox. For other websites, inspect their terms and `robots.txt`, identify yourself appropriately, and avoid excessive request rates.


In [1]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"

RATING_WORDS = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}

OUTPUT_COLUMNS = [
    "UPC", "Title", "Price (£)", "Rating",
    "Genre", "Availability", "Description"
]

print("Imports and constants are ready.")


Imports and constants are ready.


## 2. Small parsing helpers

Separating parsing into helpers makes the main function easier to read and test.

- `parse_price()` uses a regular expression to extract the numeric part of a price such as `£17.93` and returns a `float`.
- `parse_rating()` reads the word embedded in a class such as `star-rating Four` and converts it to the number `4`.
- `clean_text()` collapses tabs, line breaks, and repeated spaces into a single space.


In [2]:
def clean_text(value):
    # Collapse repeated whitespace and safely handle a missing element.
    if value is None:
        return ""
    text = value.get_text(" ", strip=True) if hasattr(value, "get_text") else str(value)
    return " ".join(text.split())


def parse_price(price_text):
    # Convert a displayed pound price into a float.
    match = re.search(r"\d+(?:\.\d+)?", price_text)
    if not match:
        raise ValueError(f"Could not parse price from {price_text!r}")
    return float(match.group())


def parse_rating(rating_element):
    # Convert a class such as 'star-rating Four' into integer 4.
    if rating_element is None:
        raise ValueError("Rating element was not found")
    for class_name in rating_element.get("class", []):
        if class_name in RATING_WORDS:
            return RATING_WORDS[class_name]
    raise ValueError(f"Unknown rating classes: {rating_element.get('class', [])}")


## 3. Parse one product-detail page

The product table uses header/value rows, so a dictionary comprehension turns it into a convenient lookup such as `product_information["UPC"]`.

The genre is the final linked breadcrumb before the current book title. The description is selected with `#product_description + p`, meaning “the paragraph immediately following the element whose ID is `product_description`.” If no description exists, an empty string is returned.


In [3]:
def parse_product_page(session, product_url, timeout=20):
    # Download and extract detail-only fields for one book.
    response = session.get(product_url, timeout=timeout)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    product_information = {
        row.select_one("th").get_text(strip=True): row.select_one("td").get_text(strip=True)
        for row in soup.select("table.table.table-striped tr")
    }

    breadcrumb_links = soup.select("ul.breadcrumb li a")
    genre = clean_text(breadcrumb_links[-1]) if breadcrumb_links else ""

    description_element = soup.select_one("#product_description + p")
    description = clean_text(description_element)

    availability_text = product_information.get(
        "Availability",
        clean_text(soup.select_one("p.availability"))
    )
    availability = "In stock" if "In stock" in availability_text else "Out of stock"

    return {
        "UPC": product_information.get("UPC", ""),
        "Genre": genre,
        "Availability": availability,
        "Description": description,
    }


## 4. Main `scrape_books` function

The function validates its arguments before sending requests:

- ratings must be from 1 through 5;
- maximum price cannot be negative.

For each listing page, `article.product_pod` selects all book cards. A book is retained when `rating >= min_rating` **and** `price <= max_price`. The next-page URL is taken from `li.next a`, so the code does not hard-code the number of pages.

`try/finally` guarantees that the HTTP session closes even if an error occurs. A small delay is added between product requests as polite scraping practice.


In [4]:
def scrape_books(min_rating, max_price):
    # Scrape qualifying books and return the requested pandas DataFrame.
    if isinstance(min_rating, bool) or not isinstance(min_rating, (int, float)):
        raise TypeError("min_rating must be a number")
    if isinstance(max_price, bool) or not isinstance(max_price, (int, float)):
        raise TypeError("max_price must be a number")
    if not 1 <= min_rating <= 5:
        raise ValueError("min_rating must be between 1 and 5")
    if max_price < 0:
        raise ValueError("max_price cannot be negative")

    records = []
    listing_url = BASE_URL
    session = requests.Session()
    session.headers.update({
        "User-Agent": "EducationalWebScrapingLab/1.0"
    })

    try:
        page_number = 0
        while listing_url is not None:
            page_number += 1
            response = session.get(listing_url, timeout=20)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            book_cards = soup.select("article.product_pod")
            if not book_cards:
                raise RuntimeError(
                    f"No book cards found on {listing_url}; the page structure may have changed."
                )

            for card in book_cards:
                price = parse_price(clean_text(card.select_one("p.price_color")))
                rating = parse_rating(card.select_one("p.star-rating"))

                # Filter before visiting the detail page.
                if rating < min_rating or price > max_price:
                    continue

                link = card.select_one("h3 a")
                if link is None or not link.get("href"):
                    continue

                title = link.get("title") or clean_text(link)
                product_url = urljoin(listing_url, link["href"])
                details = parse_product_page(session, product_url)

                records.append({
                    "UPC": details["UPC"],
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": details["Genre"],
                    "Availability": details["Availability"],
                    "Description": details["Description"],
                })

                time.sleep(0.05)

            next_link = soup.select_one("li.next a")
            listing_url = (
                urljoin(listing_url, next_link["href"])
                if next_link is not None
                else None
            )

            print(
                f"Processed listing page {page_number}; "
                f"qualifying books collected: {len(records)}",
                end="\r"
            )
    finally:
        session.close()

    print()
    return pd.DataFrame(records, columns=OUTPUT_COLUMNS)


## 5. Execute the requested search

The lab asks for books rated **4.0 or higher** and costing **£20 or less**. Both limits are inclusive because the function uses `rating < min_rating` and `price > max_price` only to reject records.

The assertions are reproducibility checks: they prove that the result has the requested schema and that every returned row satisfies both filters.


In [5]:
books = scrape_books(min_rating=4.0, max_price=20)

display(books)
print(f"Number of qualifying books: {len(books)}")

assert books.columns.tolist() == OUTPUT_COLUMNS
assert books["Rating"].ge(4.0).all()
assert books["Price (£)"].le(20).all()
assert books["UPC"].ne("").all()

books.to_csv("books_rating_4plus_price_20_or_less.csv", index=False)
print("Saved books_rating_4plus_price_20_or_less.csv")


Processed listing page 50; qualifying books collected: 75


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock,Aaron Ledbetterâs future had been planned ou...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock,"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock,A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock,"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock,"Every summer, Rose goes with her mom and dad t..."
...,...,...,...,...,...,...,...
70,9c96cd1329fbd82d,The Zombie Room,19.69,5,Default,In stock,An unlikely bond is forged between three men f...
71,b78deb463531d078,The Silent Wife,12.34,5,Fiction,In stock,A chilling psychological thriller about a marr...
72,4280ac3eab57aa5d,The Girl You Lost,12.29,5,Mystery,In stock,Eighteen years ago your baby daughter was snat...
73,29fc016c459aeb14,The Edge of Reason (Bridget Jones #2),19.18,4,Womens Fiction,In stock,Monday 27 Januaryâ7:15 a.m. Hurrah! The wild...


Number of qualifying books: 75
Saved books_rating_4plus_price_20_or_less.csv


## 6. Quick analysis and validation

The first table summarizes price and rating. The genre count shows which genres appear most often among the qualifying books. These observations describe this sandbox catalogue only; the site states that its prices and ratings are randomly assigned and have no real-world meaning.


In [6]:
display(books[["Price (£)", "Rating"]].describe())

genre_counts = books["Genre"].value_counts().rename("number_of_books")
display(genre_counts)

print("Duplicate UPCs:", books["UPC"].duplicated().sum())
print("Missing descriptions:", books["Description"].eq("").sum())
print("Availability values:", sorted(books["Availability"].unique()))

assert books["UPC"].is_unique


,Price (£),Rating
count,75.000000,75.00000
mean,14.570933,4.56000
std,2.728553,0.49973
min,10.000000,4.00000
25%,12.340000,4.00000
50%,14.440000,5.00000
75%,16.850000,5.00000
max,19.690000,5.00000


,number_of_books
Genre,
Default,10
Sequential Art,9
Young Adult,7
Nonfiction,6
Fiction,5
Poetry,4
Food and Drink,4
Fantasy,3
Romance,3


Duplicate UPCs: 0
Missing descriptions: 0
Availability values: ['In stock']


## 7. Experiment with other filters

The function is reusable. For example, this request five-star books costing at most £30.


In [8]:
five_star_under_30 = scrape_books(min_rating=5, max_price=30)
display(five_star_under_30.head())
print(len(five_star_under_30))


Processed listing page 50; qualifying books collected: 78


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock,Aaron Ledbetterâs future had been planned ou...
1,c2e46a2ee3b4a322,Chase Me (Paris Nights #2),25.27,5,Romance,In stock,"A Michelin two-star chef at twenty-eight, Viol..."
2,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock,"In The Four Agreements, don Miguel Ruiz reveal..."
3,5dada2b7be26bd03,The Elephant Tree,23.82,5,Thriller,In stock,Mark Fallon is an overworked detective investi...
4,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock,A page-turning novel that is also an explorati...


78
